# NMX Reduction Workflow

> NMX does not expect users to use python interface directly.<br>
This documentation is mostly for instrument data scientists or instrument scientists.<br>

## TL;DR

In [1]:
from ess.nmx.executables import reduction
from ess.nmx.data import get_small_nmx_nexus
from ess.nmx.configurations import (
    ReductionConfig,
    OutputConfig,
    InputConfig,
    WorkflowConfig,
    TimeBinCoordinate,
)

# Build Configuration
config = ReductionConfig(
    inputs=InputConfig(
        input_file=[get_small_nmx_nexus().as_posix()],
        detector_ids=[0, 1, 2],
    ),
    output=OutputConfig(
        output_file="scipp_output.hdf", skip_file_output=False, overwrite=True
    ),
    workflow=WorkflowConfig(
        time_bin_coordinate=TimeBinCoordinate.time_of_flight,
        nbins=10,
        tof_simulation_num_neutrons=1_000_000,
        tof_simulation_min_wavelength=1.8,
        tof_simulation_max_wavelength=3.6,
        tof_simulation_seed=42,
    ),
)

# Run reduction and display the result.
result = reduction(config=config, display=display)
dg = result.to_datagroup()
dg

Unzipping contents of '/home/runner/.cache/ess/nmx/small_nmx_nexus.hdf.zip' to '/home/runner/.cache/ess/nmx/small_nmx_nexus.hdf.zip.unzip'


'Input file: /home/runner/.cache/ess/nmx/small_nmx_nexus.hdf.zip.unzip/small_nmx_nexus.hdf'

'Output file: /home/runner/work/ess/ess/packages/essnmx/docs/user-guide/scipp_output.hdf'

/home/runner/work/ess/ess/packages/essnmx/src/ess/nmx/workflows.py:190: RuntimeWarning: No crystal rotation found in the Nexus file under 'entry/sample/crystal_rotation'. Returning zero rotation.
  warnings.warn(


DataGroup(sizes={'y_pixel_offset': 1280, 'x_pixel_offset': 1280, 'tof': 10, 'distance': 204, 'event_time_offset': 287}, keys=[
    control: DataGroup(4, {'y_pixel_offset': 1280, 'x_pixel_offset': 1280, 'tof': 10}),
    definitions: NXlauetof,
    instrument: DataGroup(3, {'y_pixel_offset': 1280, 'x_pixel_offset': 1280, 'tof': 10}),
    sample: DataGroup(6, {}),
    lookup_table: DataGroup(6, {'distance': 204, 'event_time_offset': 287}),
    reducer: DataGroup(1, {}),
])

## Configuration

`essnmx` provides a command line data reduction tool.<br>
The `essnmx-reduce` interface will reduce `nexus` file <br>
and save the results into `NXlauetof`(not exactly but very close) format for `dials`.<br>

For conveniences and safety, all configuration options are wrapped in a nested pydantic model.<br>
Here is a python API you can use to build the configuration and turn it into command line arguments.

**The configuration object is a pydantic model, and it thus enforces strict checks on the types of the arguments.**

In [2]:
from ess.nmx.configurations import (
    ReductionConfig,
    OutputConfig,
    InputConfig,
    WorkflowConfig,
    TimeBinCoordinate,
    Compression,
    to_command_arguments,
)

config = ReductionConfig(
    inputs=InputConfig(
        input_file=["PATH_TO_THE_NEXUS_FILE.hdf"],
        detector_ids=[0, 1, 2],  # Detector index to be reduced in alphabetical order.
    ),
    output=OutputConfig(output_file="scipp_output.hdf", skip_file_output=True),
    workflow=WorkflowConfig(
        time_bin_coordinate=TimeBinCoordinate.time_of_flight,
        nbins=10,
        tof_simulation_num_neutrons=1_000_000,
        tof_simulation_min_wavelength=1.8,
        tof_simulation_max_wavelength=3.6,
        tof_simulation_seed=42,
    ),
)

display(config)
print(to_command_arguments(config=config, one_line=True))

ReductionConfig(inputs=InputConfig(input_file=['PATH_TO_THE_NEXUS_FILE.hdf'], swmr=False, detector_ids=[0, 1, 2], iter_chunk=False, chunk_size_pulse=0, chunk_size_events=0), workflow=WorkflowConfig(time_bin_coordinate=<TimeBinCoordinate.time_of_flight: 'time_of_flight'>, time_bin_width=None, nbins=10, min_time_bin=None, max_time_bin=None, time_bin_unit=<TimeBinUnit.ms: 'ms'>, lookup_table_file_path=None, tof_simulation_num_neutrons=1000000, tof_simulation_min_wavelength=1.8, tof_simulation_max_wavelength=3.6, tof_simulation_min_ltotal=150.0, tof_simulation_max_ltotal=170.0, tof_simulation_seed=42), output=OutputConfig(verbose=False, skip_file_output=True, output_file='scipp_output.hdf', overwrite=False, compression=<Compression.BITSHUFFLE_LZ4: 'BITSHUFFLE_LZ4'>))

--input-file PATH_TO_THE_NEXUS_FILE.hdf \
--detector-ids 0 1 2 \
--chunk-size-pulse 0 \
--chunk-size-events 0 \
--time-bin-coordinate time_of_flight \
--nbins 10 \
--time-bin-unit ms \
--tof-simulation-num-neutrons 1000000 \
--tof-simulation-min-wavelength 1.8 \
--tof-simulation-max-wavelength 3.6 \
--tof-simulation-min-ltotal 150.0 \
--tof-simulation-max-ltotal 170.0 \
--tof-simulation-seed 42 \
--skip-file-output \
--output-file scipp_output.hdf \
--compression BITSHUFFLE_LZ4


## Reduce Nexus File(s)

`OutputConfig` has an option called `skip_file_output` if you want to reduce the file and use it only on the memory.<br>
Then you can use `save_results` function to explicitly save the results.

In [3]:
from ess.nmx.executables import reduction
from ess.nmx.data import get_small_nmx_nexus

config = ReductionConfig(
    inputs=InputConfig(input_file=[get_small_nmx_nexus().as_posix()]),
    output=OutputConfig(skip_file_output=True),
)
results = reduction(config=config, display=display)
results

'Input file: /home/runner/.cache/ess/nmx/small_nmx_nexus.hdf.zip.unzip/small_nmx_nexus.hdf'

'Output file: /home/runner/work/ess/ess/packages/essnmx/docs/user-guide/scipp_output.h5'

/home/runner/work/ess/ess/packages/essnmx/src/ess/nmx/workflows.py:190: RuntimeWarning: No crystal rotation found in the Nexus file under 'entry/sample/crystal_rotation'. Returning zero rotation.
  warnings.warn(


NMXLauetof(control=NMXMonitorMetadata(data=<scipp.DataArray>
Dimensions: Sizes[y_pixel_offset:1280, x_pixel_offset:1280, tof:23, ]
Coordinates:
* tof                       float64            [µs]  (tof [bin-edge])  [73744.3, 76744.3, ..., 139744, 142744]
Data:
                            float32         [counts]  (y_pixel_offset, x_pixel_offset, tof)  [1, 1, ..., 1, 1]  [1, 1, ..., 1, 1]

, tof_bin_coord='tof', mode=<ControlMode.monitor: 'monitor'>, preset=<scipp.Variable> ()    float64         [counts]  0), definitions='NXlauetof', instrument=NMXInstrument(detectors=DataGroup(sizes={}, keys=[
    detector_panel_0: NMXReducedDetector(data=<scipp.DataArray>
Dimensions: Sizes[y_pixel_offset:1280, x_pixel_offset:1280, tof:23, ]
Coordinates:
  Ltotal                    float64              [m]  (y_pixel_offset, x_pixel_offset)  [157.209, 157.208, ..., 157.208, 157.208]
* detector_number             int32        <no unit>  (y_pixel_offset, x_pixel_offset)  [0, 1, ..., 1638398, 1638399]
* po

In [4]:
from ess.nmx.executables import save_results

output_config = OutputConfig(
    output_file="scipp_output.hdf", overwrite=True, compression=Compression.GZIP
)
save_results(results=results, output_config=output_config)

## Loading Reduced File

There is a custom loader for NXlauetof file for NMX.<br>
It reconstructs the position coordinates from the file and adds them back to the data array.<br>
The data group should almost look the same as the in-memory results.<br>
The loaded data group will not have some coordinates compared to the in-memory results.<br>

In [5]:
from ess.nmx._nxlauetof_io import load_essnmx_nxlauetof

loaded = load_essnmx_nxlauetof('scipp_output.hdf')
loaded

DataGroup(sizes={'y_pixel_offset': 1280, 'x_pixel_offset': 1280, 'tof': 23}, keys=[
    control: DataGroup(4, {'y_pixel_offset': 1280, 'x_pixel_offset': 1280, 'tof': 23}),
    definitions: NXlauetof,
    instrument: DataGroup(3, {'y_pixel_offset': 1280, 'x_pixel_offset': 1280, 'tof': 23}),
    reducer: DataGroup(1, {}),
    sample: DataGroup(6, {}),
])

You can then plot the loaded data array exactly same as the in-memory results.

For example, you can plot the 3D instrument view:

```python
%matplotlib widget
import scipp as sc
import scippneutron as scn


dims=('y_pixel_offset', 'x_pixel_offset')
merged_2d_das = sc.concat(
    [
        det['data'].sum('tof').flatten(dims=dims, to='detector_number')
        for det in loaded['instrument']['detectors'].values()
    ],
    dim='detector_number',
)

scn.instrument_view(merged_2d_das)
```

## Compression Modes

There are multiple compression modes for `detector counts` data(other datasets are not compressed).<br>
The default mode is `BITSHUFFLE_LZ4`.<br>

Here is the rough benchmark results with the small test dataset.<br>
With the result, users can decide which compression mode to use.

| Compression Mode | Final Size [MB] | Compression Ratio | Writing Time [s] | Reading Time [s] |
| ---------------- | --------------- |------------------ | ---------------- | ---------------- |
| NONE | 1_966 | 1 | 4 | 1 |
| GZIP | 5 | 370 | 18 | 5 |
| BITSHUFFLE_LZ4 | 17 | 114 | 10 | 3 |

> In the ESS standard VISA environment. (64 GB mem/6 VCPUs)

`BITSHUFFLE_LZ4` showed almost twice faster speed for writing/reading the reduced file.<br>
`GZIP` and `BITSHUFFLE` could both compress the data more than 99% (when the histogram was very empty)<br>
but `GZIP` had 3 times better compression ratio than `BITSHUFFLE` for this particular dataset.<br>

.. note::
Why `BITSHUFFLE` is the default compression mode? -
`Bitshuffle` is compatible with `DIALS` and other crystallography packages.
It is the primary compression mode of all data collected on `DECTRIS EIGER` detectors,
which are the primary detectors used at synchrotron X-ray MX beamlines.
Most of these packages can also read `gzip` data but the slow readout makes `gzip` less attractive than `bitshuffle`.


.. warning::
`Bitshuffle` may not be supported in cerntain environments, such as MacOS or Windows.
It was accepted to be default because the first step of the reduction workflow (this workflow)
is expected to be run by ESS in the specific environment that bitshuffle supports.